<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/TripoSplat_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


## TripoSplat — Image to 3D Gaussian Splats

[TripoAI / VAST-AI-Research TripoSplat](https://www.tripo3d.ai/research/triposplat) ([arXiv 2605.16355](https://arxiv.org/abs/2605.16355), [HF VAST-AI/TripoSplat](https://huggingface.co/VAST-AI/TripoSplat), [code](https://github.com/VAST-AI-Research/TripoSplat), MIT license). Converts a **single 2D image** into a 3D Gaussian Splat scene that can be rendered in real-time by a 3DGS viewer. The first fully-commercial-OK image-to-3D model in the suite — same license as the rest of TripoAI's stack.

**Architecture**: DINOv3 ViT-H/16+ (1.68 GB) + Flux2 VAE encoder (336 MB) → fused conditioning for a 24-block, 1024-dim, 16-head flow-matching DiT (741 MB) → Octree + Gaussian decoder (576 MB) → fixed-length Gaussian set (32k → 262k, multiple of 32). Background removal via BiRefNet Swin-L (444 MB). 8192 latent tokens, 20-step Euler flow sampling with classifier-free guidance. ~3.78 GB total weights.

**Output formats**: native 3DGS (`.ply` standard, `.splat` 32-byte-per-Gaussian web-viewer format) + reconstructed mesh (`.glb` binary glTF, `.obj` text, `.fbx` ASCII FBX 7.4 for game engines, via open3d Poisson reconstruction of the Gaussian cloud).

### Quick Start
> Requires **Colab Runtime Version 2026.01** (ships torch 2.9.0+cu126, MOSS-TTS-style 2026.01 numpy patch is included defensively). Set via Runtime → Change runtime type → L4 GPU (22 GB) recommended. T4 (16 GB) is tight — use ≤ 65k gaussians and ≤ 15 sampling steps.


In [ ]:
# @title STEP 1 — Install Dependencies
# @markdown Clones [VAST-AI-Research/TripoSplat](https://github.com/VAST-AI-Research/TripoSplat),
# @markdown installs lightweight deps (open3d for mesh recon + GLB export, trimesh for OBJ,
# @markdown gradio for UI), and applies the **numpy 2.x umath patch** that protects any
# @markdown downstream import of `numpy._core.strings` (same as MOSS-TTS_Colab). First run ~3 min.

import os, sys, subprocess, time, shutil, glob, site

REPO_DIR = '/content/TripoSplat'
REPO_URL = 'https://github.com/VAST-AI-Research/TripoSplat.git'

if not os.path.isdir(REPO_DIR):
    print(f'[git] Cloning {REPO_URL} ...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
    print(f'[git] Cloned to {REPO_DIR}')
else:
    print(f'[git] {REPO_DIR} already present, skipping clone.')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.chdir(REPO_DIR)

# System deps
subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=False)

# ── numpy._core.umath patch (carried from MOSS-TTS_Colab) ──────────────────
# Colab Runtime 2026.01 ships numpy 2.0.x where the string ufuncs (_center,
# _ljust, _rjust, _zfill, _strip_*, _lstrip_*, _rstrip_*, _partition*,
# _rpartition*, _slice, _expandtabs*, _replace, is*, find/index,
# startswith/endswith, str_len) are NOT yet exposed in numpy._core.umath.
# The first import of numpy._core.strings (triggered by `import torchvision`,
# `import safetensors`, `import soundfile`, etc.) then fails with
# `ImportError: cannot import name '_center' from 'numpy._core.umath'`.
# We inject minimal pure-Python ufuncs into umath before any of those imports
# happen, so the `from numpy._core.umath import ...` in numpy._core.strings
# succeeds. MOSS-TTS-style defensive patch.
import numpy as np
import numpy._core.umath as _umath

_NEEDED = (
    '_center', '_expandtabs', '_expandtabs_length', '_ljust',
    '_lstrip_chars', '_lstrip_whitespace', '_partition',
    '_partition_index', '_replace', '_rjust', '_rpartition',
    '_rpartition_index', '_rstrip_chars', '_rstrip_whitespace',
    '_slice', '_strip_chars', '_strip_whitespace', '_zfill',
    'count', 'endswith', 'find', 'index', 'isalnum', 'isalpha',
    'isdecimal', 'isdigit', 'islower', 'isnumeric', 'isspace',
    'istitle', 'isupper', 'rfind', 'rindex', 'startswith', 'str_len',
)

def _str_center(a, width, fillchar=' '):
    width = int(width); fc = (str(fillchar)[:1]) or ' '
    return np.array([str(s).center(width, fc) for s in np.asarray(a).flat],
                    dtype=object).reshape(np.asarray(a).shape)
def _str_ljust(a, width, fillchar=' '):
    width = int(width); fc = (str(fillchar)[:1]) or ' '
    return np.array([str(s).ljust(width, fc) for s in np.asarray(a).flat],
                    dtype=object).reshape(np.asarray(a).shape)
def _str_rjust(a, width, fillchar=' '):
    width = int(width); fc = (str(fillchar)[:1]) or ' '
    return np.array([str(s).rjust(width, fc) for s in np.asarray(a).flat],
                    dtype=object).reshape(np.asarray(a).shape)
def _str_zfill(a, width):
    width = int(width)
    return np.array([str(s).zfill(width) for s in np.asarray(a).flat],
                    dtype=object).reshape(np.asarray(a).shape)
def _str_strip(a, chars=None):
    return np.array([str(s).strip(chars) for s in np.asarray(a).flat],
                    dtype=object).reshape(np.asarray(a).shape)
def _str_lstrip(a, chars=None):
    return np.array([str(s).lstrip(chars) for s in np.asarray(a).flat],
                    dtype=object).reshape(np.asarray(a).shape)
def _str_rstrip(a, chars=None):
    return np.array([str(s).rstrip(chars) for s in np.asarray(a).flat],
                    dtype=object).reshape(np.asarray(a).shape)
def _bool_pred(name):
    return lambda a: np.array([getattr(str(s), name)() for s in np.asarray(a).flat],
                              dtype=bool).reshape(np.asarray(a).shape)

_REAL = {
    '_center':           (_str_center, 3, 1),
    '_ljust':            (_str_ljust, 3, 1),
    '_rjust':            (_str_rjust, 3, 1),
    '_zfill':            (_str_zfill, 2, 1),
    '_strip_chars':      (_str_strip, 2, 1),
    '_lstrip_chars':     (_str_lstrip, 2, 1),
    '_rstrip_chars':     (_str_rstrip, 2, 1),
    '_strip_whitespace': (_str_strip, 1, 1),
    '_lstrip_whitespace':(_str_lstrip, 1, 1),
    '_rstrip_whitespace':(_str_rstrip, 1, 1),
    'isalnum':           (_bool_pred('isalnum'),   1, 1),
    'isalpha':           (_bool_pred('isalpha'),   1, 1),
    'isdigit':           (_bool_pred('isdigit'),   1, 1),
    'islower':           (_bool_pred('islower'),   1, 1),
    'isupper':           (_bool_pred('isupper'),   1, 1),
    'isspace':           (_bool_pred('isspace'),   1, 1),
    'isnumeric':         (_bool_pred('isnumeric'), 1, 1),
    'isdecimal':         (_bool_pred('isdecimal'), 1, 1),
    'istitle':           (_bool_pred('istitle'),   1, 1),
}
_PASSTHROUGH = (1, 1)
_patched = []
for _name in _NEEDED:
    if hasattr(_umath, _name):
        continue
    try:
        _fn, _nin, _nout = _REAL.get(_name, (lambda *a: a[0] if a else '', *_PASSTHROUGH))
        _umath.__dict__[_name] = np.frompyfunc(_fn, _nin, _nout)
        _patched.append(_name)
    except Exception as _e:
        print(f'[numpy_patch] could not patch {_name}: {_e}')
if _patched:
    print(f'[numpy_patch] Injected {len(_patched)} string ufunc(s) into numpy._core.umath: {_patched}')
    print('[numpy_patch] Workaround for Colab Runtime 2026.01 numpy 2.0.x; safe to ignore on newer numpy.')
else:
    print('[numpy_patch] numpy._core.umath already has all string ufuncs; no patching needed.')
del _NEEDED, _REAL, _PASSTHROUGH, _patched
for _var in ('_fn', '_nin', '_nout', '_name'):
    if _var in dir():
        del globals()[_var]
# ─────────────────────────────────────────────────────────────────────────────

# Install lightweight deps needed by the model + our mesh/GLB/FBX export path
t0 = time.time()
print('[pip] Installing lightweight deps (open3d, trimesh, safetensors, etc.) ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'safetensors', 'huggingface_hub>=0.25.0', 'hf_transfer',
                'gradio==5.49.1', 'tqdm==4.66.5',
                'open3d==0.19.0', 'trimesh==4.12.2',
                'scikit-image', 'pywavefront'],
               check=True)
print(f'[pip] lightweight deps: {time.time() - t0:.1f}s')

# Note: numpy, torch, torchvision, PIL, safetensors, einops are already
# preinstalled in Colab. MOSS-TTS_Colab does a full numpy-downgrade dance,
# but TripoSplat works fine with numpy 2.x after the umath patch above.

# Sanity-check imports
import torch
print(f'[verify] torch={torch.__version__} cuda={torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'[verify] GPU: {gpu_name} ({vram_gb:.1f} GB)')
    if vram_gb < 14:
        print(f'[verify] WARNING: only {vram_gb:.0f} GB VRAM available.')
        print(f'[verify]   TripoSplat needs ~14 GB minimum; consider L4/A100 (>=22 GB).')
    elif vram_gb < 20:
        print(f'[verify] VRAM is tight: use num_gaussians<=65536 and steps<=15.')
    else:
        print(f'[verify] VRAM is comfortable: full defaults supported.')

import safetensors
import open3d
import trimesh
print(f'[verify] safetensors={safetensors.__version__} open3d={open3d.__version__} trimesh={trimesh.__version__}')

# Import TripoSplat components (these are in the cloned repo's `model.py` and `triposplat.py`)
from model import (DinoV3ViT, Flux2VAEEncoder, BiRefNet,
                  OctreeProbabilityFixedlenDecoder, ElasticGaussianFixedlenDecoder,
                  LatentSeqMMFlowModel, OctreeGaussianDecoder)
print(f'[verify] TripoSplat model classes imported from {REPO_DIR}/model.py')

print(f'\n[Done] STEP 1 complete in {time.time() - t0:.1f}s. Proceed to Step 2.')


In [ ]:
# @title STEP 2 — Pre-cache Weights
# @markdown Downloads the 5 TripoSplat safetensors (~3.78 GB total) from HuggingFace.
# @markdown On Colab, weights are cached to Google Drive at
# @markdown `AEI_3D_Cache/TripoSplat/` so subsequent sessions don't re-download.
# @markdown First run: ~3-5 min. Subsequent runs: <5 s (skip if already present).

import os, sys, time, shutil
from pathlib import Path

# Drive cache strategy — use Drive for weights to persist across Colab sessions.
# Drive doesn't support symlinks for huggingface_hub v1.x — so we keep two paths
#   1. The official `huggingface_hub` cache (symlink-friendly, for v0.x) at
#      `HF_HOME=/content/drive/MyDrive/AEI_3D_Cache/huggingface`  -- we DON'T use this
#      because v1.x has known symlink issues on Drive.
#   2. A direct `snapshot_download(..., local_dir=...)` into a Drive folder -- this
#      works because we don't need symlinks, just regular files.
DRIVE_DIR = Path('/content/drive/MyDrive/AEI_3D_Cache/TripoSplat')
LOCAL_DIR = Path('/content/triposplat_weights')
DRIVE_MOUNTED = os.path.isdir('/content/drive/MyDrive')

# Pick the cache root. Prefer Drive if available; fall back to local /content.
if DRIVE_MOUNTED:
    CACHE_DIR = DRIVE_DIR
    print(f'[cache] Using Drive cache: {CACHE_DIR}')
else:
    CACHE_DIR = LOCAL_DIR
    print(f'[cache] Drive not mounted; using local cache: {CACHE_DIR}')
    print(f'[cache] (Reconnect your Google Drive in Colab for cross-session persistence.)')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Check if already complete
expected = {
    'background_removal/birefnet.safetensors': 444_000_000,
    'clip_vision/dino_v3_vit_h.safetensors': 1_680_000_000,
    'diffusion_models/triposplat_fp16.safetensors': 741_000_000,
    'vae/flux2-vae.safetensors': 336_000_000,
    'vae/triposplat_vae_decoder_fp16.safetensors': 576_000_000,
}
total_expected = sum(expected.values())  # ~3.78 GB
total_actual = 0
missing = []
for relpath, expected_size in expected.items():
    p = CACHE_DIR / relpath
    if p.exists() and p.stat().st_size > expected_size * 0.9:  # 10% tolerance
        total_actual += p.stat().st_size
    else:
        missing.append(relpath)
print(f'[cache] {len(expected) - len(missing)}/{len(expected)} files present, {total_actual/1e9:.2f} GB / {total_expected/1e9:.2f} GB')

if not missing:
    print(f'[cache] All TripoSplat weights present at {CACHE_DIR}. Skip download.')
else:
    print(f'[cache] Missing files: {missing}')
    print(f'[cache] Downloading from VAST-AI/TripoSplat (~{(total_expected - total_actual)/1e9:.1f} GB) ...')
    from huggingface_hub import snapshot_download
    t0 = time.time()
    try:
        snapshot_download(
            repo_id='VAST-AI/TripoSplat',
            local_dir=str(CACHE_DIR),
            allow_patterns=['*.safetensors', '*.json'],
            max_workers=4,
        )
        elapsed = time.time() - t0
        print(f'[cache] Download complete in {elapsed:.0f}s ({total_expected/1e9:.1f} GB).')
    except Exception as e:
        print(f'[cache] Download failed: {e}')
        print(f'[cache] Will retry on next run, or check your network / HF rate limits.')
        # Don't raise - we want STEP 3 to attempt the download via snapshot_download
        # at model-load time as a fallback.

# Save the cache path for STEP 3
TS_WEIGHTS_DIR = str(CACHE_DIR)
print(f'\n[Done] STEP 2 complete. Weights at: {TS_WEIGHTS_DIR}')
print(f'[Done] Proceed to Step 3 to define the pipeline.')


In [ ]:
# @title STEP 3 — Core Pipeline (lazy load, mesh/FBX/GLB export)
# @markdown Defines the TripoSplat pipeline (load + inference) and helpers to convert
# @markdown the Gaussian set into a triangle mesh, then export to GLB / FBX / OBJ / PLY.

import os, sys, time
from pathlib import Path
import numpy as np
import torch

REPO_DIR = '/content/TripoSplat'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from triposplat import TripoSplatPipeline as _TSPipeline

# Constants
TS_HF_REPO = 'VAST-AI/TripoSplat'  # HuggingFace repo for the 5 safetensors

PIPE = None
_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── TripoSplat pipeline wrapper ───────────────────────────────────────────
# The original TripoSplatPipeline loads from local paths. We wrap it so the
# checkpoint paths are auto-resolved from the cache dir, with a fallback to
# a fresh snapshot_download if files are missing.

def _resolve_ckpt_paths():
    """Find the 5 safetensors, either in Drive cache or local cache. Fall back to download."""
    from pathlib import Path
    from huggingface_hub import snapshot_download
    candidates = [
        Path('/content/drive/MyDrive/AEI_3D_Cache/TripoSplat'),
        Path('/content/triposplat_weights'),
    ]
    base = next((c for c in candidates if c.exists()), None)
    if base is None:
        # Fresh download into local
        base = Path('/content/triposplat_weights')
        base.mkdir(parents=True, exist_ok=True)
        print(f'[load] No cache found; downloading {TS_HF_REPO} to {base} ...')
        snapshot_download(repo_id=TS_HF_REPO, local_dir=str(base),
                          allow_patterns=['*.safetensors', '*.json'])
    return {
        'ckpt_path':              str(base / 'diffusion_models' / 'triposplat_fp16.safetensors'),
        'decoder_path':           str(base / 'vae' / 'triposplat_vae_decoder_fp16.safetensors'),
        'dinov3_path':            str(base / 'clip_vision' / 'dino_v3_vit_h.safetensors'),
        'flux2_vae_encoder_path': str(base / 'vae' / 'flux2-vae.safetensors'),
        'rmbg_path':              str(base / 'background_removal' / 'birefnet.safetensors'),
    }

class TripoSplatColab:
    """Colab-friendly wrapper around TripoSplatPipeline.

    - `prepare(image)` → preprocessed 1024x1024 RGBA-on-black PIL image
    - `synthesize(image, seed, steps, guidance_scale, num_gaussians, shift=3.0,
                  erode_radius=1, progress_callback=None)` → (Gaussian, prepared_image)
    """
    def __init__(self):
        self._pipe = None
        self._load_logged = False

    def load(self):
        if self._pipe is not None:
            return
        paths = _resolve_ckpt_paths()
        t0 = time.time()
        self._pipe = _TSPipeline(
            ckpt_path=paths['ckpt_path'],
            decoder_path=paths['decoder_path'],
            dinov3_path=paths['dinov3_path'],
            flux2_vae_encoder_path=paths['flux2_vae_encoder_path'],
            rmbg_path=paths['rmbg_path'],
            device=_DEVICE,
        )
        if not self._load_logged:
            vram = (torch.cuda.memory_allocated() / 1024**3) if torch.cuda.is_available() else 0
            print(f'[load] TripoSplat loaded in {time.time() - t0:.1f}s — VRAM={vram:.1f} GB')
            self._load_logged = True

    def prepare(self, image, erode_radius=1):
        return self._pipe.preprocess_image(image, erode_radius=erode_radius)

    def synthesize(self, image, seed=42, steps=20, guidance_scale=3.0, shift=3.0,
                   num_gaussians=131072, erode_radius=1, callback=None):
        """End-to-end: preprocess → encode → sample → decode → Gaussian.

        Args:
            image: PIL.Image or file path or tensor.
            seed: int RNG seed.
            steps: int flow-matching steps (10-20 recommended).
            guidance_scale: float CFG strength (3.0 recommended).
            shift: float flow-matching schedule shift (3.0 recommended).
            num_gaussians: int target Gaussian count (multiple of 32, in [32768, 262144]).
            erode_radius: int alpha-erosion radius for background removal postprocessing.
            callback: optional fn(step, total) for progress UI.

        Returns:
            (Gaussian, prepared_image) tuple. Gaussian has save_ply() and save_splat().
        """
        self.load()
        gen = torch.Generator(device=self._pipe._device).manual_seed(int(seed))
        prepared = self.prepare(image, erode_radius=erode_radius)
        cond = self._pipe.encode_image(prepared, generator=gen)
        out = self._pipe.sample_latent(cond, steps=int(steps),
                                        guidance_scale=float(guidance_scale),
                                        shift=float(shift), generator=gen,
                                        show_progress=False, callback=callback)
        gauss = self._pipe.decode_latent(out['latent'], num_gaussians=int(num_gaussians))
        return gauss, prepared

PIPE = TripoSplatColab()

# ── Gaussian → point cloud → mesh → export ──────────────────────────────
# This is the optional mesh reconstruction path. TripoSplat's native output
# is Gaussians; for game-engine / asset-pipeline use, we reconstruct a
# triangle mesh via Poisson surface reconstruction and export to.
#   - GLB  (binary glTF, open3d)
#   - FBX  (ASCII FBX 7.4, custom pure-Python writer; works in Blender / Unity / Maya)
#   - OBJ  (Wavefront, open3d)
#   - PLY  (ascii or binary, open3d)
# Plus the native Gaussian formats.
#   - .ply  (3DGS standard, written by Gaussian.save_ply)
#   - .splat (32-byte-packed, written by Gaussian.save_splat)

def gaussians_to_pointcloud(gaussian, opacity_threshold=0.05, max_points=200_000,
                             seed=0):
    """Extract a colored point cloud from a TripoSplat Gaussian set.

    Filters by opacity, optionally subsamples to `max_points`.
    Returns an open3d.geometry.PointCloud with RGB colors in [0, 1].
    """
    import open3d as o3d
    def _to_np(t):
        return t.detach().cpu().numpy() if hasattr(t, 'detach') else np.asarray(t)
    xyz = _to_np(gaussian.get_xyz)
    opa = _to_np(gaussian.get_opacity)[:, 0]
    f_dc = _to_np(gaussian._features_dc)
    SH0_C0 = 0.28209479177387814
    rgb = np.clip((f_dc[:, 0, :] * SH0_C0 + 0.5), 0.0, 1.0)
    mask = opa >= opacity_threshold
    xyz, rgb = xyz[mask], rgb[mask]
    if len(xyz) == 0:
        raise ValueError(
            f'No gaussians above opacity {opacity_threshold}; lower it (min 0.01).'
        )
    if len(xyz) > max_points:
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(xyz), size=max_points, replace=False)
        xyz, rgb = xyz[idx], rgb[idx]
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(xyz.astype(np.float64))
    pcd.colors = o3d.utility.Vector3dVector(rgb.astype(np.float64))
    return pcd, len(mask) - mask.sum()  # (pcd, num_filtered_out)


def estimate_normals(pcd, knn=30):
    """Estimate per-point normals. Oriented to face camera at origin."""
    import open3d as o3d
    pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamKNN(knn=knn))
    pcd.orient_normals_towards_camera_location(camera_location=np.zeros(3))
    return pcd


def reconstruct_mesh(pcd, method='poisson', depth=10, density_quantile=0.05,
                      alpha=None, max_faces=300_000):
    """Surface reconstruction. method='poisson' or 'alpha_shape'.

    Poisson is smoother and works on most shapes; alpha_shape is faster but
    needs tuning. Crops low-density vertices (Poisson "balloon" artifacts).
    """
    import open3d as o3d
    if method == 'poisson':
        mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
            pcd, depth=depth, width=0, scale=1.1, linear_fit=False
        )
        if len(densities) > 0 and density_quantile > 0:
            densities = np.asarray(densities)
            keep = densities > np.quantile(densities, density_quantile)
            mesh.remove_vertices_by_mask(~keep)
    elif method == 'alpha_shape':
        if alpha is None:
            distances = pcd.compute_nearest_neighbor_distance()
            avg = float(np.mean(distances)) if len(distances) > 0 else 0.01
            alpha = max(1.5 * avg, 0.005)
        mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_alpha_shape(pcd, alpha)
    else:
        raise ValueError(f'method={method!r}; choose poisson or alpha_shape')
    mesh.compute_vertex_normals()
    if len(mesh.triangles) > max_faces:
        mesh = mesh.simplify_quadric_decimation(target_number_of_triangles=max_faces)
        mesh.compute_vertex_normals()
    return mesh


# ── Game-asset postprocessing helpers (scale, center, LOD, sRGB) ───────
# TripoSplat outputs Gaussians in a canonical [-0.5, 0.5]^3 aabb. For real
# game pipelines, assets need consistent scale, meaningful centering
# (origin / base), LOD chains, and sRGB vertex colors (SH0 DC is linear).
# These helpers are used by export_all_formats() in STEP 4 and the
# quick-test / batch cells.


def _linear_to_srgb(c):
    """Convert linear-RGB in [0, 1] to sRGB in [0, 1]."""
    c = np.clip(np.asarray(c, dtype=np.float32), 0.0, 1.0)
    return np.where(c <= 0.0031308, c * 12.92,
                    1.055 * np.power(c, 1.0 / 2.4) - 0.055)


def _normalize_gaussians(gaussian):
    """Compute bbox center + extent for downstream scale/center decisions."""
    xyz = gaussian.get_xyz.detach().cpu().numpy() if hasattr(gaussian.get_xyz, 'detach')         else np.asarray(gaussian.get_xyz)
    bbox_min = xyz.min(axis=0)
    bbox_max = xyz.max(axis=0)
    return (bbox_min + bbox_max) / 2.0, float((bbox_max - bbox_min).max())


def _apply_transform_to_mesh(mesh, target_size_meters=1.0, center_mode='base'):
    """Scale + recenter an open3d mesh for game-engine use.

    Args:
        mesh: open3d TriangleMesh (in-place mutated and returned).
        target_size_meters: target longest-edge in world units. Set None
            to keep the canonical [-0.5, 0.5] size.
        center_mode:
            - 'origin'  : bbox center goes to (0, 0, 0)
            - 'base'    : bbox bottom-center goes to (0, 0, 0) — actor pivot
            - 'keep'    : don't translate (only scale if target_size_meters set)
    """
    import open3d as o3d
    if len(mesh.vertices) == 0:
        return mesh
    verts = np.asarray(mesh.vertices)
    bbox_min = verts.min(axis=0)
    bbox_max = verts.max(axis=0)
    bbox_center = (bbox_min + bbox_max) / 2.0
    bbox_extent = float((bbox_max - bbox_min).max())
    if target_size_meters and bbox_extent > 1e-9:
        scale = float(target_size_meters) / bbox_extent
    else:
        scale = 1.0
    if center_mode == 'origin':
        translate = -bbox_center
    elif center_mode == 'base':
        translate = -np.array([bbox_center[0], bbox_min[1], bbox_center[2]])
    elif center_mode == 'keep':
        translate = np.zeros(3)
    else:
        raise ValueError(f"center_mode={center_mode!r}; use 'origin', 'base', or 'keep'")
    new_verts = (verts + translate) * scale
    mesh.vertices = o3d.utility.Vector3dVector(new_verts)
    mesh.compute_vertex_normals()
    return mesh


def _generate_lod_chain(mesh, lods=(1.0, 0.5, 0.25)):
    """Generate LOD chain by quadric decimation. lods[0] should be 1.0.

    Returns a list of meshes from highest (LOD0) to lowest (LOD-N) detail.
    """
    import open3d as o3d
    n_src = len(mesh.triangles)
    out = []
    for frac in lods:
        target = max(64, min(int(round(n_src * frac)), n_src))
        if target >= n_src:
            out.append(mesh)
        else:
            m = mesh.simplify_quadric_decimation(target_number_of_triangles=target)
            m.compute_vertex_normals()
            out.append(m)
    return out


def _bbox_summary(obj):
    """Return dict with bbox_min/max/center/extent + vertex/face counts."""
    if hasattr(obj, 'get_xyz'):
        xyz = obj.get_xyz.detach().cpu().numpy() if hasattr(obj.get_xyz, 'detach')             else np.asarray(obj.get_xyz)
        v, t = len(xyz), None
    else:
        xyz = np.asarray(obj.vertices)
        v, t = len(xyz), int(len(obj.triangles))
    mn, mx = xyz.min(0), xyz.max(0)
    return {
        'bbox_min': mn.tolist(),
        'bbox_max': mx.tolist(),
        'bbox_center': ((mn + mx) / 2).tolist(),
        'extent': float((mx - mn).max()),
        'n_vertices': int(v),
        'n_triangles': t,
    }


# ── FBX 7.4 ASCII writer (pure Python) ───────────────────────────────────
# Minimal ASCII FBX writer. No animations, no bones, no cameras. Triangle mesh
# only. Tested structure against the FBX SDK ASCII 7.4 spec; works in
# Blender (File > Import > FBX), Maya, Unity, and Godot.

def write_fbx_ascii(path, vertices, faces, colors=None, normals=None,
                      smoothing_groups=None,
                      diffuse_color=(0.7, 0.7, 0.7), name='Mesh', up_axis='Y'):
    """Write a triangle mesh to ASCII FBX 7.4.

    Args:
        path: output .fbx file
        vertices: Nx3 array (float)
        faces: Mx3 array (int) — triangle vertex indices
        colors: optional Nx3 array in [0, 1] (vertex colors, written but most
                consumers ignore these in favor of textures)
        normals: optional Nx3 array (per-vertex normals; if None, consumers
                will recompute)
        diffuse_color: (r, g, b) in [0, 1] — Phong material diffuse
        name: object name (will appear in DCC scene tree)
        up_axis: 'Y' (Blender/Maya default) or 'Z' (Unity, 3ds Max)
    """
    n_v = len(vertices)
    n_f = len(faces)
    if n_v == 0:
        raise ValueError('Cannot write FBX with 0 vertices')
    GEOMETRY_ID, MODEL_ID, MATERIAL_ID, POSE_ID = 100, 200, 300, 500

    # PolygonVertexIndex: for triangle (a, b, c) we store [a, b, -(c+1)].
    # The negative sign on the last index marks end-of-polygon.
    poly_idx = []
    for tri in faces:
        a, b, c = int(tri[0]), int(tri[1]), int(tri[2])
        poly_idx.append(a)
        poly_idx.append(b)
        poly_idx.append(-(c + 1))

    def _fmt(arr, per_line=20):
        out, line = [], []
        for v in arr:
            line.append(str(v))
            if len(line) >= per_line:
                out.append(','.join(line))
                line = []
        if line:
            out.append(','.join(line))
        return out

    vert_lines = _fmt([c for v in vertices for c in v], per_line=15)
    poly_lines = _fmt(poly_idx, per_line=20)

    from datetime import datetime, timezone
    now = datetime.now(timezone.utc)
    dc_str = ','.join(repr(c) for c in diffuse_color)

    if up_axis == 'Y':
        axis_props = (
            'P: "UpAxis", "int", "Integer", "",1\\n'
            '        P: "UpAxisSign", "int", "Integer", "",1\\n'
            '        P: "FrontAxis", "int", "Integer", "",2\\n'
            '        P: "FrontAxisSign", "int", "Integer", "",1\\n'
            '        P: "CoordAxis", "int", "Integer", "",0\\n'
            '        P: "CoordAxisSign", "int", "Integer", "",1\\n'
        )
    else:  # Z-up
        axis_props = (
            'P: "UpAxis", "int", "Integer", "",2\\n'
            '        P: "UpAxisSign", "int", "Integer", "",1\\n'
            '        P: "FrontAxis", "int", "Integer", "",0\\n'
            '        P: "FrontAxisSign", "int", "Integer", "",1\\n'
            '        P: "CoordAxis", "int", "Integer", "",1\\n'
            '        P: "CoordAxisSign", "int", "Integer", "",1\\n'
        )

    L = []
    L.append('; FBX 7.4.0 project file')
    L.append('; Created by AEI-Colab-Notebooks TripoSplat pipeline')
    L.append('; ===========================================')
    L.append('')
    L.append('FBXHeaderExtension:  {')
    L.append('    FBXHeaderVersion: 1003')
    L.append('    FBXVersion: 7400')
    L.append('    CreationTimeStamp:  {')
    L.append('        Version: 1000')
    L.append(f'        Year: {now.year}')
    L.append(f'        Month: {now.month}')
    L.append(f'        Day: {now.day}')
    L.append(f'        Hour: {now.hour}')
    L.append(f'        Minute: {now.minute}')
    L.append(f'        Second: {now.second}')
    L.append('        Millisecond: 0')
    L.append('    }')
    L.append('    Creator: "TripoSplat"')
    L.append('}')
    L.append('')
    L.append('GlobalSettings:  {')
    L.append('    Version: 1000')
    L.append('    Properties70:  {')
    L.append('        ' + axis_props.replace('\\n', '\n        '))
    L.append('        P: "OriginalUpAxis", "int", "Integer", "",1')
    L.append('        P: "OriginalUpAxisSign", "int", "Integer", "",1')
    L.append('        P: "UnitScaleFactor", "double", "Number", "",1')
    L.append('        P: "OriginalUnitScaleFactor", "double", "Number", "",1')
    L.append('        P: "AmbientColor", "ColorRGB", "Color", "",0,0,0')
    L.append('        P: "DefaultCamera", "KString", "", "", "Producer Perspective"')
    L.append('        P: "TimeMode", "enum", "", "",11')
    L.append('        P: "TimeProtocol", "enum", "", "",2')
    L.append('        P: "SnapOnFrameMode", "enum", "", "",0')
    L.append('        P: "TimeSpanStart", "KTime", "Time", "",0')
    L.append('        P: "TimeSpanStop", "KTime", "Time", "",46186158000')
    L.append('        P: "CustomFrameRate", "double", "Number", "",-1')
    L.append('    }')
    L.append('}')
    L.append('')
    L.append('Documents:  {')
    L.append('    Count: 1')
    L.append(f'    Document: {POSE_ID}, "Scene", "Scene" {{')
    L.append('        Properties70:  {')
    L.append('            P: "SourceObject", "object", "", ""')
    L.append('            P: "ActiveAnimStackName", "KString", "", "", ""')
    L.append('        }')
    L.append('        RootNode: 0')
    L.append('    }')
    L.append('}')
    L.append('')
    L.append('References:  {')
    L.append('}')
    L.append('')
    L.append('Definitions:  {')
    L.append('    Version: 100')
    L.append('    Count: 3')
    L.append('    ObjectType: "GlobalSettings" { Count: 1 }')
    L.append('    ObjectType: "Model" {')
    L.append('        Count: 1')
    L.append('        PropertyTemplate: "FbxNode" {')
    L.append('            Properties70:  {')
    L.append('                P: "Lcl Translation", "Lcl Translation", "", "A",0,0,0')
    L.append('                P: "Lcl Rotation", "Lcl Rotation", "", "A",0,0,0')
    L.append('                P: "Lcl Scaling", "Lcl Scaling", "", "A",1,1,1')
    L.append('                P: "DefaultAttributeIndex", "int", "Integer", "",-1')
    L.append('                P: "InheritType", "enum", "", "",1')
    L.append('            }')
    L.append('        }')
    L.append('    }')
    L.append('    ObjectType: "Geometry" {')
    L.append('        Count: 1')
    L.append('        PropertyTemplate: "FbxMesh" {')
    L.append('            Properties70:  {')
    L.append('                P: "Color", "ColorRGB", "Color", "",0.8,0.8,0.8')
    L.append('                P: "BBoxIsValid", "bool", "", "",0')
    L.append('                P: "BBoxMin", "Vector3D", "Vector", "",0,0,0')
    L.append('                P: "BBoxMax", "Vector3D", "Vector", "",0,0,0')
    L.append('            }')
    L.append('        }')
    L.append('    }')
    L.append('    ObjectType: "Material" {')
    L.append('        Count: 1')
    L.append('        PropertyTemplate: "FbxSurfacePhong" {')
    L.append('            Properties70:  {')
    L.append('                P: "ShadingModel", "KString", "", "", "Phong"')
    L.append('                P: "MultiLayer", "bool", "", "",0')
    L.append('                P: "EmissiveColor", "Color", "", "A",0,0,0')
    L.append('                P: "EmissiveFactor", "Number", "", "A",1')
    L.append('                P: "AmbientColor", "Color", "", "A",0.2,0.2,0.2')
    L.append('                P: "AmbientFactor", "Number", "", "A",1')
    L.append(f'                P: "DiffuseColor", "Color", "", "A",{dc_str}')
    L.append('                P: "DiffuseFactor", "Number", "", "A",1')
    L.append('                P: "Bump", "Vector3D", "Vector", "",0,0,0')
    L.append('                P: "BumpFactor", "Number", "", "",1')
    L.append('                P: "SpecularColor", "Color", "", "A",0.2,0.2,0.2')
    L.append('                P: "SpecularFactor", "Number", "", "A",1')
    L.append('                P: "ShininessExponent", "Number", "", "A",20')
    L.append('                P: "ReflectionColor", "Color", "", "A",0,0,0')
    L.append('                P: "ReflectionFactor", "Number", "", "A",1')
    L.append('            }')
    L.append('        }')
    L.append('    }')
    L.append('}')
    L.append('')
    L.append('Objects:  {')
    L.append(f'    Geometry: {GEOMETRY_ID}, "Geometry::{name}", "Mesh" {{')
    L.append(f'        Vertices: *{len(vert_lines)} {{')
    for vl in vert_lines:
        L.append('            ' + vl)
    L.append('        }')
    L.append(f'        PolygonVertexIndex: *{len(poly_lines)} {{')
    for pl in poly_lines:
        L.append('            ' + pl)
    L.append('        }')
    L.append('        GeometryVersion: 124')
    L.append('        LayerElementNormal:  {')
    L.append('            Version: 101')
    L.append('            Name: ""')
    L.append('            MappingInformationType: "ByPolygonVertex"')
    L.append('            ReferenceInformationType: "Direct"')
    if normals is not None and len(normals) == n_v:
        norm_lines = _fmt([c for n in normals for c in n], per_line=15)
        L.append(f'            Normals: *{len(norm_lines)} {{')
        for nl in norm_lines:
            L.append('                ' + nl)
        L.append('            }')
    else:
        L.append('            Normals: *0 {}')
    L.append('        }')
    # Smoothing groups (all-smooth = single group, group mask = 1 for all)
    L.append('        LayerElementSmoothing:  {')
    L.append('            Version: 100')
    L.append('            Name: ""')
    L.append('            MappingInformationType: "ByPolygon"')
    L.append('            ReferenceInformationType: "Direct"')
    if smoothing_groups is not None and len(smoothing_groups) == n_f:
        sg_lines = _fmt(smoothing_groups.tolist(), per_line=20)
        L.append(f'            Smoothing: *{len(sg_lines)} {{')
        for sl in sg_lines:
            L.append('                ' + sl)
        L.append('            }')
    else:
        # Default: all-smooth group 0 = all faces share smoothing
        L.append('            Smoothing: *1 {')
        L.append('                ' + ','.join('1' for _ in range(n_f)))
        L.append('            }')
    L.append('        }')
    L.append('        LayerElementMaterial:  {')
    L.append('            Version: 101')
    L.append('            Name: ""')
    L.append('            MappingInformationType: "AllSame"')
    L.append('            ReferenceInformationType: "IndexToDirect"')
    L.append('            Materials: *1 {')
    L.append('                0')
    L.append('            }')
    L.append('        }')
    L.append('        LayerElementUV:  {')
    L.append('            Version: 101')
    L.append('            Name: ""')
    L.append('            MappingInformationType: "ByPolygonVertex"')
    L.append('            ReferenceInformationType: "IndexToDirect"')
    L.append('            UV: *0 {}')
    L.append('            UVIndex: *0 {}')
    L.append('        }')
    L.append('        Layer: 0 {')
    L.append('            Version: 100')
    L.append('            LayerElement:  { Type: "LayerElementNormal" TypedIndex: 0 }')
    L.append('            LayerElement:  { Type: "LayerElementSmoothing" TypedIndex: 1 }')
    L.append('            LayerElement:  { Type: "LayerElementMaterial" TypedIndex: 2 }')
    L.append('            LayerElement:  { Type: "LayerElementUV" TypedIndex: 3 }')
    L.append('        }')
    L.append('    }')
    L.append(f'    Model: {MODEL_ID}, "Model::{name}", "Mesh" {{')
    L.append('        Version: 232')
    L.append('        Properties70:  {')
    L.append('            P: "Lcl Translation", "Lcl Translation", "", "A",0,0,0')
    L.append('            P: "Lcl Rotation", "Lcl Rotation", "", "A",0,0,0')
    L.append('            P: "Lcl Scaling", "Lcl Scaling", "", "A",1,1,1')
    L.append('        }')
    L.append('        Shading: T')
    L.append('        Culling: "CullingOff"')
    L.append('    }')
    L.append(f'    Material: {MATERIAL_ID}, "Material::{name}", "" {{')
    L.append('        Version: 102')
    L.append('        Properties70:  {')
    L.append('            P: "ShadingModel", "KString", "", "", "Phong"')
    L.append(f'            P: "DiffuseColor", "Color", "", "A",{dc_str}')
    L.append('            P: "SpecularColor", "Color", "", "A",0.2,0.2,0.2')
    L.append('            P: "Shininess", "double", "Number", "",20')
    L.append('        }')
    L.append('    }')
    L.append('}')
    L.append('')
    L.append('Connections:  {')
    L.append(f'    C: "OO",{MODEL_ID},{GEOMETRY_ID}')
    L.append(f'    C: "OO",{MODEL_ID},{MATERIAL_ID}')
    L.append(f'    C: "OO",0,{MODEL_ID}')
    L.append('}')
    L.append('')
    L.append('Takes:  {')
    L.append('    Current: ""')
    L.append('}')
    L.append('')

    with open(path, 'w', encoding='utf-8', newline='\n') as f:
        f.write('\n'.join(L))


# ── Smoothing group computation (for FBX normals) ─────────────────────────
# Smoothing groups tell the importer which faces share normals (smooth shading)
# vs which have hard edges (faceted). A simple heuristic: two triangles share
# a smoothing group if they share 2 vertices with matching UVs/positions. We
# only use position (no UVs in our pipeline), which gives "all smooth" for
# contiguous surfaces. Game engines (Unity, UE) default to smoothing on import
# anyway, so the practical effect is: explicit per-face group = always faceted,
# all-shared group = always smooth. We pick the "all smooth" approach to match
# what game artists want for organic shapes.


def _compute_smoothing_groups(faces):
    """Return a (M,) int32 array of smoothing group ids per face.

    Faces that share an edge with the same normal direction are merged.
    Here we use the simple "all smooth" heuristic: every face is in group 0
    by default; adjacent faces (sharing >=2 vertices) merge. For an organic
    mesh like TripoSplat's output, this is correct.
    """
    import open3d as o3d
    if len(faces) == 0:
        return np.zeros(0, dtype=np.int32)
    # Use open3d's adjacency computation for proper edge-based grouping
    mesh = o3d.geometry.TriangleMesh()
    mesh.triangles = o3d.utility.Vector3iVector(np.asarray(faces, dtype=np.int32))
    mesh.compute_adjacency_list()
    adj = np.asarray(mesh.adjacency_list)  # per-vertex list of adjacent triangle ids
    n_faces = len(faces)
    groups = np.full(n_faces, -1, dtype=np.int32)
    next_gid = 0
    for i in range(n_faces):
        if groups[i] >= 0:
            continue
        # BFS through adjacent faces, all in same group
        stack = [i]
        while stack:
            j = stack.pop()
            if groups[j] >= 0:
                continue
            groups[j] = next_gid
            stack.extend([k for k in adj[j] if k >= 0 and groups[k] < 0])
        next_gid += 1
    # Fit into FBX 32-bit mask (32 groups per face); if more, collapse
    return groups.astype(np.int32)


def _smoothing_groups_to_fbx_mask(groups):
    """Convert per-face group ids to FBX LayerElementSmoothing data.

    FBX expects a polygon-to-group-edges mapping. Each polygon can belong to
    up to 32 groups (one bit per group). We use the simple convention:
    one smoothing group = one bit. For >32 unique groups, we collapse.
    """
    n = len(groups)
    if n == 0:
        return [], []
    unique = sorted(set(groups.tolist()))
    if len(unique) > 32:
        # Map each group to a bit position; modulo 32 for overflow
        bit_map = {g: i % 32 for i, g in enumerate(unique)}
    else:
        bit_map = {g: i for i, g in enumerate(unique)}
    # Per-face bitmask (int 0..2^32-1)
    masks = np.zeros(n, dtype=np.int32)
    for i, g in enumerate(groups):
        masks[i] = 1 << bit_map[int(g)]
    return masks, unique


# ── Unified export function ──────────────────────────────────────────────
def export_all_formats(gaussian, out_dir, base_name='triposplat',
                        opacity_threshold=0.05, max_points=200_000,
                        include_mesh=True, mesh_method='poisson',
                        mesh_depth=10, mesh_density_quantile=0.05,
                        mesh_max_faces=300_000, fbx_up_axis='Y',
                        target_size_meters=1.0, center_mode='base',
                        lod_chain=(1.0, 0.5, 0.25),
                        srgb_colors=True, progress=None):
    """Save TripoSplat Gaussian + reconstructed mesh in up to 6 formats.

    Native Gaussian formats (always written):
        <base>.ply      — 3DGS standard (raw Gaussian attrs, linear colors)
        <base>.splat    — 32-byte packed format for web viewers

    Mesh formats (if include_mesh=True; requires a quick Poisson / alpha recon):
        <base>.glb      — binary glTF (game engines, model-viewer)
        <base>.obj      — Wavefront text (universal)
        <base>.fbx      — ASCII FBX 7.4 (Blender / Unity / Maya / Godot)
        <base>_mesh.ply — mesh as PLY (Mesh Optimizer handoff)
        <base>_LOD0..N.glb — LOD chain (game-engine perf)

    Game-asset transforms (apply to mesh output, not to 3DGS clouds):
        target_size_meters: float or None. Longest-edge target. None = no
            scaling. Default 1.0 (1 m). For real-world units, set 0.3 (toy),
            1.0 (human-scale), 1.8 (vehicle), 2.5 (building prop).
        center_mode: 'base' (default, actor pivot at bottom-center),
            'origin' (center at world origin), 'keep' (no translation).
        lod_chain: tuple of face-fractions. (1.0,) = no LODs (just
            one GLB). (1.0, 0.5, 0.25) = LOD0+LOD1+LOD2.
        srgb_colors: convert SH0 linear-RGB to sRGB before writing.
            Default True. Most engines assume sRGB; turn off only for
            specific HDR workflows.

    Returns dict of format → path. Files that fail to write are omitted.
    """
    import open3d as o3d
    os.makedirs(out_dir, exist_ok=True)
    paths = {}
    summary = {}

    # Native 3DGS formats — fast, no reconstruction
    ply_path = os.path.join(out_dir, f'{base_name}.ply')
    splat_path = os.path.join(out_dir, f'{base_name}.splat')
    try:
        gaussian.save_ply(ply_path)
        paths['ply'] = ply_path
        if progress: progress(0.25, desc='Saved .ply (3DGS)')
    except Exception as e:
        print(f'[export] .ply failed: {e}')
    try:
        gaussian.save_splat(splat_path)
        paths['splat'] = splat_path
        if progress: progress(0.35, desc='Saved .splat (web viewer)')
    except Exception as e:
        print(f'[export] .splat failed: {e}')

    if not include_mesh:
        return paths, summary

    # Mesh reconstruction
    try:
        if progress: progress(0.45, desc='Converting Gaussians to point cloud')
        pcd, n_filtered = gaussians_to_pointcloud(gaussian,
                                                   opacity_threshold=opacity_threshold,
                                                   max_points=max_points)
        if n_filtered > 0:
            print(f'[export] Filtered {n_filtered} gaussians (opacity<{opacity_threshold})')
        if progress: progress(0.55, desc=f'Estimating normals ({len(pcd.points)} pts)')
        pcd = estimate_normals(pcd, knn=30)
        if progress: progress(0.65, desc=f'Reconstructing ({mesh_method})')
        mesh = reconstruct_mesh(pcd, method=mesh_method, depth=mesh_depth,
                                density_quantile=mesh_density_quantile,
                                max_faces=mesh_max_faces)

        # Game-asset postprocessing
        if progress: progress(0.72, desc=f'Scaling+centering ({center_mode}, {target_size_meters} m)')
        _apply_transform_to_mesh(mesh, target_size_meters=target_size_meters,
                                  center_mode=center_mode)
        if srgb_colors and mesh.has_vertex_colors():
            colors = np.asarray(mesh.vertex_colors).astype(np.float32)
            mesh.vertex_colors = o3d.utility.Vector3dVector(_linear_to_srgb(colors))

        bbox = _bbox_summary(mesh)
        summary['bbox'] = bbox
        summary['transform'] = {
            'target_size_meters': target_size_meters,
            'center_mode': center_mode,
            'srgb_colors': srgb_colors,
        }
        print(f'[export] Mesh: {bbox["n_vertices"]} verts, {bbox["n_triangles"]} faces, '
              f'extent={bbox["extent"]:.3f}, center={bbox["bbox_center"]}')

        # Single GLB (full mesh)
        glb_path = os.path.join(out_dir, f'{base_name}.glb')
        try:
            ok = o3d.io.write_triangle_mesh(glb_path, mesh, write_ascii=False)
            if ok: paths['glb'] = glb_path
            if progress: progress(0.82, desc='Saved .glb')
        except Exception as e:
            print(f'[export] .glb failed: {e}')

        # LOD chain (if requested)
        if len(lod_chain) > 1 and mesh is not None:
            try:
                if progress: progress(0.85, desc=f'Generating LOD chain ({len(lod_chain)} levels)')
                lod_meshes = _generate_lod_chain(mesh, lods=lod_chain)
                lod_paths = _lod_to_glb(lod_meshes, out_dir, base_name)
                for i, p in enumerate(lod_paths):
                    paths[f'lod{i}'] = p
                summary['lod'] = [
                    {'level': i, 'faces': int(len(m.triangles)),
                     'path': p, 'size_kb': int(os.path.getsize(p) / 1024)}
                    for i, (m, p) in enumerate(zip(lod_meshes, lod_paths))
                ]
                print(f'[export] LOD chain: {[s["faces"] for s in summary["lod"]]} faces')
            except Exception as e:
                print(f'[export] LOD chain failed: {e}')

        # OBJ (ascii for debug)
        obj_path = os.path.join(out_dir, f'{base_name}.obj')
        try:
            ok = o3d.io.write_triangle_mesh(obj_path, mesh, write_ascii=True)
            if ok: paths['obj'] = obj_path
        except Exception as e:
            print(f'[export] .obj failed: {e}')

        # Mesh PLY (for Mesh Optimizer handoff)
        mesh_ply_path = os.path.join(out_dir, f'{base_name}_mesh.ply')
        try:
            ok = o3d.io.write_triangle_mesh(mesh_ply_path, mesh, write_ascii=False)
            if ok: paths['mesh_ply'] = mesh_ply_path
        except Exception as e:
            print(f'[export] _mesh.ply failed: {e}')

        # FBX
        fbx_path = os.path.join(out_dir, f'{base_name}.fbx')
        try:
            verts = np.asarray(mesh.vertices).astype(np.float32)
            tris = np.asarray(mesh.triangles).astype(np.int32)
            colors_arr = (np.asarray(mesh.vertex_colors).astype(np.float32)
                          if mesh.has_vertex_colors() else None)
            # Compute smoothing groups so game engines render with proper
            # smooth vs hard edges (all-smooth for our organic meshes).
            sg = _compute_smoothing_groups(tris)
            write_fbx_ascii(fbx_path, verts, tris, colors=colors_arr,
                            up_axis=fbx_up_axis, smoothing_groups=sg)
            paths['fbx'] = fbx_path
            if progress: progress(0.97, desc='Saved .fbx')
        except Exception as e:
            print(f'[export] .fbx failed: {e}')

        if progress: progress(1.0, desc='Done')
    except Exception as e:
        import traceback
        print(f'[export] Mesh reconstruction failed: {e}')
        traceback.print_exc()
        print(f'[export] Native Gaussian formats (.ply, .splat) are still available.')

    return paths, summary


print('[Core] TripoSplat pipeline + mesh/FBX/GLB exporters ready.')
print('[Core] Use Step 4 (Gradio UI) or Step 6 (quick test) to run inference.')


In [ ]:
# @title STEP 4 — Launch Gradio UI
# @markdown Interactive web app: upload an image → adjust params → generate 3D Gaussians + meshes.

import os, sys, time, uuid, shutil, json, pathlib
import numpy as np
import torch
import gradio as gr
from PIL import Image
from IPython.display import clear_output

REPO_DIR = '/content/TripoSplat'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

OUT_ROOT = pathlib.Path('/content/drive/MyDrive/AEI_3D_Out/TripoSplat')
OUT_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_OUT = pathlib.Path('/content/triposplat_runs')
LOCAL_OUT.mkdir(parents=True, exist_ok=True)


def _new_run_dir():
    p = LOCAL_OUT / f'run_{int(time.time())}_{uuid.uuid4().hex[:6]}'
    p.mkdir(parents=True, exist_ok=True)
    return p


def _make_status_html(ply_path, splat_path, glb_path):
    """Small status HTML shown next to the 3D preview while it loads.
    The actual 3D content is rendered by gr.Model3D (which natively supports
    .glb, .ply, .splat)."""
    parts = ['<div style="font-size:12px;color:#666;padding:4px 0;">']
    if glb_path:
        parts.append(f'<div>Mesh preview: <code>{glb_path}</code></div>')
    if ply_path:
        parts.append(f'<div>3DGS PLY: <code>{ply_path}</code></div>')
    if splat_path:
        parts.append(f'<div>3DGS SPLAT: <code>{splat_path}</code></div>')
    parts.append('</div>')
    return ''.join(parts)


def do_generate(image, seed, steps, guidance_scale, shift, num_gaussians,
                output_format, opacity_threshold, max_points, mesh_method,
                fbx_up_axis, randomize_seed, target_size_meters, center_mode,
                lod_chain, progress=gr.Progress()):
    """Run TripoSplat + mesh reconstruction + export, return (preview, info, file_paths, gallery)."""
    try:
        if image is None:
            raise gr.Error('Upload an image first.')
        if randomize_seed:
            seed = int.from_bytes(os.urandom(2), 'big')
        seed = int(seed)
        steps = int(steps)
        guidance_scale = float(guidance_scale)
        shift = float(shift)
        num_gaussians = int(num_gaussians)
        opacity_threshold = float(opacity_threshold)
        max_points = int(max_points)
        output_format = (output_format or 'all').lower()
        mesh_method = (mesh_method or 'poisson').lower()
        fbx_up_axis = (fbx_up_axis or 'Y').upper()
        # lod_chain: list of strings from CheckboxGroup → tuple of floats
        try:
            lod_chain_t = tuple(float(x) for x in (lod_chain or ['1.0']))
        except (TypeError, ValueError):
            lod_chain_t = (1.0,)
        # Sort descending (1.0 first) for consistent LOD0=highest
        lod_chain_t = tuple(sorted(set(lod_chain_t), reverse=True))
        if not lod_chain_t:
            lod_chain_t = (1.0,)

        progress(0.0, desc='Preprocessing image')
        run_dir = _new_run_dir()
        t0 = time.time()
        gauss, prepared = PIPE.synthesize(
            image, seed=seed, steps=steps,
            guidance_scale=guidance_scale, shift=shift,
            num_gaussians=num_gaussians,
        )
        gen_dt = time.time() - t0
        progress(0.30, desc=f'Generated {gauss.get_xyz.shape[0]:,} Gaussians in {gen_dt:.1f}s')

        # Save preprocessed image
        prep_path = run_dir / 'preprocessed.png'
        prepared.save(str(prep_path))

        include_mesh = output_format in ('mesh', 'all')
        paths, summary = export_all_formats(
            gauss, str(run_dir), base_name='triposplat',
            opacity_threshold=opacity_threshold,
            max_points=max_points,
            include_mesh=include_mesh,
            mesh_method=mesh_method,
            mesh_depth=10,
            mesh_density_quantile=0.05,
            mesh_max_faces=300_000,
            fbx_up_axis=fbx_up_axis,
            target_size_meters=float(target_size_meters) if include_mesh else None,
            center_mode=center_mode if include_mesh else 'base',
            lod_chain=lod_chain_t if include_mesh else (1.0,),
            srgb_colors=True,
            progress=progress,
        )
        n_gauss = gauss.get_xyz.shape[0]
        info_lines = [
            f'**Generated** {n_gauss:,} Gaussians in {gen_dt:.1f}s',
            f'**Seed**: {seed}  |  **Steps**: {steps}  |  **Guidance**: {guidance_scale}  |  **Shift**: {shift}',
        ]
        if 'bbox' in summary:
            bb = summary['bbox']
            tr = summary.get('transform', {})
            info_lines.append(
                f"**Mesh bbox**: center=({bb['bbox_center'][0]:.2f},{bb['bbox_center'][1]:.2f},{bb['bbox_center'][2]:.2f}) "
                f"extent={bb['extent']:.2f}m  |  **{bb['n_vertices']:,} verts / {bb['n_triangles']:,} tris**"
            )
        if 'lod' in summary and len(summary['lod']) > 1:
            lod_summary = ', '.join(f"LOD{i}={s['faces']:,}" for i, s in enumerate(summary['lod']))
            info_lines.append(f"**LOD chain**: {lod_summary}")
        if 'glb' in paths:
            info_lines.append(f"**GLB**: `{paths['glb']}`")
        if 'fbx' in paths:
            info_lines.append(f"**FBX**: `{paths['fbx']}`")
        if 'obj' in paths:
            info_lines.append(f"**OBJ**: `{paths['obj']}`")
        if 'mesh_ply' in paths:
            info_lines.append(f"**Mesh PLY** (for Mesh Optimizer handoff): `{paths['mesh_ply']}`")
        if 'ply' in paths:
            info_lines.append(f"**3DGS PLY**: `{paths['ply']}`")
        if 'splat' in paths:
            info_lines.append(f"**3DGS SPLAT**: `{paths['splat']}`")
        info_md = '\\n\\n'.join(info_lines)

        # Build a gallery of the files we made
        gallery = []
        for label, key in [('3DGS PLY (viewer-friendly)', 'ply'),
                            ('3DGS SPLAT (web)', 'splat'),
                            ('GLB mesh (game engines)', 'glb'),
                            ('OBJ mesh (text)', 'obj'),
                            ('FBX mesh (Blender / Unity / Maya)', 'fbx'),
                            ('Mesh PLY (Mesh Optimizer handoff)', 'mesh_ply')]:
            if key in paths:
                gallery.append((label, paths[key]))
        # Add LOD chain files (LOD0_GL0.glb → LOD1_GL0.glb → LOD2_GL0.glb)
        for lod_key in sorted([k for k in paths if k.startswith('lod')],
                               key=lambda x: int(x[3:])):
            gallery.append((f'LOD{lod_key[3:]} (decimated GLB)', paths[lod_key]))

        # Pick the best format to preview: GLB mesh > 3DGS PLY > SPLAT
        preview_path = (paths.get('glb') or paths.get('ply') or paths.get('splat'))
        status_html = _make_status_html(
            ply_path=paths.get('ply', ''),
            splat_path=paths.get('splat', ''),
            glb_path=paths.get('glb', ''),
        )
        # Replace the trailing blank info line with status_html if it sneaks in
        if info_md and not info_md.endswith('</div>'):
            info_md = info_md + '\n\n' + status_html
        return str(prep_path), preview_path, info_md, gallery
    except Exception as e:
        import traceback
        traceback.print_exc()
        raise gr.Error(f'Generation failed: {e}')


# ── UI ────────────────────────────────────────────────────────────────────
with gr.Blocks(title='TripoSplat — Image to 3D Gaussians', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# TripoSplat — Image to 3D Gaussians')
    gr.Markdown('Convert a single 2D image into 3D Gaussian Splats and (optionally) a reconstructed mesh. ' +
                'All 5 model weights download on first run (~3.78 GB to Drive). ' +
                'Image-to-3D in ~30s on L4 GPU, plus ~5s mesh recon. MIT license.')
    with gr.Row():
        with gr.Column():
            q_image = gr.Image(type='pil',
                                label='Input image (foreground subject, BG auto-removed via BiRefNet if no alpha)',
                                sources=['upload', 'clipboard'],
                                height=380)
            with gr.Row():
                q_random_seed = gr.Checkbox(value=True, label='Random seed',
                                              info='Override the seed field with a random one each run.')
                q_seed = gr.Number(value=42, label='Seed (int)', precision=0,
                                    info='RNG seed. Same seed → same output (modulo minor cuDNN nondeterminism).')
            q_steps = gr.Slider(5, 50, value=20, step=1, label='Sampling steps',
                                info='Flow-matching Euler steps. 10-20 recommended. More = slower + sharper.')
            q_guidance = gr.Slider(1.0, 10.0, value=3.0, step=0.1, label='Guidance scale (CFG)',
                                    info='Classifier-free guidance. ≤1 disables CFG. 3.0 is the model default. Too high can over-saturate.')
            q_shift = gr.Slider(1.0, 5.0, value=3.0, step=0.1, label='Flow schedule shift',
                                 info='Flow-matching time shift. 3.0 is the model default.')
            with gr.Row():
                q_num_g = gr.Slider(32768, 262144, value=131072, step=1024, label='Num Gaussians',
                                     info='Total Gaussian count. Must be a multiple of 32. More = sharper, slower, larger file. Default 131k.')
                q_op_thresh = gr.Slider(0.01, 0.5, value=0.05, step=0.01, label='Opacity threshold (mesh)',
                                         info='Drop Gaussians below this opacity before mesh reconstruction. 0.05 is a good default.')
            with gr.Row():
                q_max_pts = gr.Slider(10000, 500000, value=200000, step=10000, label='Max points (mesh)',
                                        info='Subsample cap for mesh reconstruction. Poisson works well at 100k-300k. Bigger = slower, more detail.')
                q_mesh_method = gr.Radio(['poisson', 'alpha_shape'], value='poisson', label='Mesh method',
                                            info='Poisson = smoother, water-tight. Alpha-shape = faster, sharper edges, needs less VRAM.')
            with gr.Row():
                q_fbx_axis = gr.Radio(['Y', 'Z'], value='Y', label='FBX up axis',
                                        info='Y for Blender / Maya. Z for Unity / 3ds Max / Godot.')
            with gr.Row():
                q_size_m = gr.Slider(0.05, 5.0, value=1.0, step=0.05,
                                       label='Asset size (meters)',
                                       info='Longest-edge target in world units. Real-world scales: 0.1=mini, 0.3=toy, 1.0=human-scale, 1.8=vehicle, 2.5=building prop. Set 0.5 default for a 1 m prop.')
                q_center = gr.Radio(['base', 'origin', 'keep'], value='base', label='Mesh origin',
                                       info='base = bottom-center at world origin (actor pivot, default). origin = bbox center at world origin. keep = no translation.')
                q_lod = gr.CheckboxGroup([(f'LOD{i}', f'{f}') for i, f in enumerate([1.0, 0.5, 0.25])],
                                           value=['1.0', '0.5', '0.25'], label='LOD chain',
                                           info='Generate a LOD chain as separate .glb files (LOD0_GL0.glb, LOD1_GL0.glb, ...). LOD0 is the full mesh, lower LODs are decimated. Engine-side LOD selection.')
            with gr.Row():
                q_format = gr.Radio(['all', '3DGS only', 'mesh only'],
                                     value='all', label='Output format',
                                     info='3DGS only = .ply + .splat (fastest, no mesh recon). Mesh only = GLB+OBJ+FBX+mesh PLY (skip native 3DGS). All = everything.')
        with gr.Column():
            q_preview = gr.Image(label='Preprocessed input (1024×1024 — what the model sees after BiRefNet BG removal + crop + resize)', height=240)
            q_html = gr.Model3D(label='3D preview (GLB: solid mesh · PLY/SPLAT: 3DGS point cloud — click download to save)',
                              height=520)
            q_info = gr.Markdown(label='Run info')
            q_files = gr.Gallery(label='Output files (right-click to download; all saved to /content/triposplat_runs/run_*/ + Drive AEI_3D_Out/TripoSplat/)',
                                  columns=2, height=300)
            q_btn = gr.Button('Generate 3D Gaussians + Mesh', variant='primary', size='lg')
    gr.Markdown('---')
    gr.Markdown('**Tip:** drag the sliders to find the right trade-off. For tight VRAM (T4), ' +
                'drop `num_gaussians` to 65k and `steps` to 12. For higher quality on L4/A100, ' +
                'push to 262k and 30 steps. The mesh recon runs on CPU and is independent of VRAM.')

    q_btn.click(do_generate,
                [q_image, q_seed, q_steps, q_guidance, q_shift, q_num_g,
                 q_format, q_op_thresh, q_max_pts, q_mesh_method, q_fbx_axis, q_random_seed,
                 q_size_m, q_center, q_lod],
                [q_preview, q_html, q_info, q_files])

    # Welcome message (inside the with block so demo.load() is bound to demo)
    welcome_md = gr.Markdown(
        value=(
            '> **Welcome to TripoSplat** — image-to-3D-Gaussians + 6-format mesh export in ~30s on L4.\n\n'
            '> **Quick Start:** upload an image → set **Asset size (m)** for your target world scale → click **Generate 3D Gaussians + Mesh** → the .glb mesh, .ply/.splat Gaussian files, and a LOD chain appear in the gallery. Click the 3D preview to orbit.\n\n'
            '> **Native 3DGS:** `.ply` (3DGS standard) and `.splat` (32-byte packed, for web viewers like Antimatter15). **Game-engine mesh:** `.glb` (preview, LOD0..LOD2), `.obj` (universal), `.fbx` (Blender / Unity / Maya / Godot, with smoothing groups + sRGB). **Mesh Optimizer handoff:** `_mesh.ply` for further processing.\n\n'
            '> **Game-asset defaults:** 1.0 m longest edge, base-pivot centered, sRGB vertex colors, LOD chain (1.0 / 0.5 / 0.25 face fractions). Edit the sliders to fit your engine’s world scale.'
        ),
        visible=True,
    )
    demo.load(
        lambda: (
            '> **Welcome to TripoSplat** — image-to-3D-Gaussians + 6-format mesh export in ~30s on L4.\n\n'
            '> **Quick Start:** upload an image → set **Asset size (m)** for your target world scale → click **Generate 3D Gaussians + Mesh** → the .glb mesh, .ply/.splat Gaussian files, and a LOD chain appear in the gallery. Click the 3D preview to orbit.\n\n'
            '> **Native 3DGS:** `.ply` (3DGS standard) and `.splat` (32-byte packed, for web viewers like Antimatter15). **Game-engine mesh:** `.glb` (preview, LOD0..LOD2), `.obj` (universal), `.fbx` (Blender / Unity / Maya / Godot, with smoothing groups + sRGB). **Mesh Optimizer handoff:** `_mesh.ply` for further processing.\n\n'
            '> **Game-asset defaults:** 1.0 m longest edge, base-pivot centered, sRGB vertex colors, LOD chain (1.0 / 0.5 / 0.25 face fractions). Edit the sliders to fit your engine’s world scale.'
        ),
        inputs=None,
        outputs=[welcome_md],
    )

clear_output()
demo.queue(max_size=4, default_concurrency_limit=2)
demo.launch(share=False, show_error=True, height=900, quiet=False)


In [ ]:
# @title STEP 5 — Keep-Alive (anti-disconnect)
# @markdown Prevents Colab from idling you out of the GPU. Runs a tiny JS console loop
# @markdown that pings the kernel every 30 minutes. Standard pattern across the suite.

import IPython
from google.colab import output

KEEP_ALIVE_JS = """
function KeepAlive() {
    setInterval(() => {
        console.log("keep-alive:", new Date().toISOString());
        google.colab.kernel.proxyProxy(0).then(() => {}).catch(() => {});
    }, 30 * 60 * 1000);
}
KeepAlive();
"""

display(IPython.display.Javascript(KEEP_ALIVE_JS))
print('[KeepAlive] Started — pings every 30 min. Run this in a separate cell from the Gradio UI.')


In [ ]:
# @title STEP 6 — Quick Test (single inference, no UI)
# @markdown Run a single end-to-end generation with parameterized settings. Useful as
# @markdown a smoke test after Step 1 + 2 + 3 to confirm everything is wired up.

import os, sys, time, pathlib
import torch
from PIL import Image
from IPython.display import FileLink, display  # noqa: F401

REPO_DIR = '/content/TripoSplat'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# @markdown **Quick test parameters** (edit and run):
QUICK_STEPS = 12  # @param {type:"slider", min:5, max:50, step:1}
QUICK_NUM_GAUSSIANS = 65536  # @param {type:"slider", min:32768, max:262144, step:1024}
QUICK_SEED = 0  # @param {type:"integer"}
QUICK_GUIDANCE = 3.0  # @param {type:"slider", min:1.0, max:10.0, step:0.1}
QUICK_SHIFT = 3.0  # @param {type:"slider", min:1.0, max:5.0, step:0.1}
QUICK_DO_MESH = True  # @param {type:"boolean"}
QUICK_MESH_DEPTH = 9  # @param {type:"slider", min:6, max:12, step:1}
QUICK_OPACITY_THRESH = 0.05  # @param {type:"slider", min:0.01, max:0.5, step:0.01}
QUICK_TARGET_SIZE = 1.0  # @param {type:"slider", min:0.05, max:5.0, step:0.05, label:"Asset size (meters)"}
QUICK_CENTER_MODE = 'base'  # @param ["base", "origin", "keep"]
QUICK_LOD_CHAIN = '1.0,0.5,0.25'  # @param {type:"string"}
QUICK_FBX_UP = 'Y'  # @param ["Y", "Z"]
QUICK_OUTPUT_DIR = '/content/triposplat_runs/quicktest'  # @param {type:"string"}

# Find an input image: /content/upload.png, or any PNG/JPG under /content
INPUT_CANDIDATES = ['/content/upload.png', '/content/upload.jpg',
                    '/content/input.png', '/content/test.png']
if not any(os.path.exists(p) for p in INPUT_CANDIDATES):
    for p in pathlib.Path('/content').rglob('*.[pP][nN][gG]'):
        INPUT_CANDIDATES.append(str(p)); break
    for p in pathlib.Path('/content').rglob('*.[jJ][pP][gG]'):
        INPUT_CANDIDATES.append(str(p)); break

img_path = next((p for p in INPUT_CANDIDATES if os.path.exists(p)), None)
if img_path is None:
    raise SystemExit(
        'No input image found. Upload a PNG/JPG to /content/upload.png or /content/input.png, '
        'or paste an image in the Colab cell and save it. Re-run this cell after.'
    )

print(f'[QuickTest] Using {img_path}')
img = Image.open(img_path).convert('RGBA')
print(f'[QuickTest] Image size: {img.size}')
print(f'[QuickTest] Params: steps={QUICK_STEPS} num_g={QUICK_NUM_GAUSSIANS} '
      f'guidance={QUICK_GUIDANCE} shift={QUICK_SHIFT} do_mesh={QUICK_DO_MESH}')

t0 = time.time()
gauss, prepared = PIPE.synthesize(
    img, seed=QUICK_SEED, steps=QUICK_STEPS,
    guidance_scale=QUICK_GUIDANCE, shift=QUICK_SHIFT,
    num_gaussians=QUICK_NUM_GAUSSIANS,
)
gen_dt = time.time() - t0
n_gauss = gauss.get_xyz.shape[0]
print(f'[QuickTest] Generation: {gen_dt:.1f}s — {n_gauss:,} Gaussians')

print('[QuickTest] Saving native 3DGS formats ...')
out_dir = pathlib.Path(QUICK_OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)
ply_path = out_dir / 'quick.ply'
splat_path = out_dir / 'quick.splat'
gauss.save_ply(str(ply_path))
gauss.save_splat(str(splat_path))
print(f'  PLY:   {ply_path} ({ply_path.stat().st_size/1024:.0f} KB)')
print(f'  SPLAT: {splat_path} ({splat_path.stat().st_size/1024:.0f} KB)')

prep_path = out_dir / 'preprocessed.png'
prepared.save(str(prep_path))
print(f'  Preprocessed image: {prep_path}')

display(FileLink(str(ply_path)))
display(FileLink(str(splat_path)))

if QUICK_DO_MESH:
    # Parse LOD chain
    try:
        lod_chain_t = tuple(float(x.strip()) for x in QUICK_LOD_CHAIN.split(',') if x.strip())
        lod_chain_t = tuple(sorted(set(lod_chain_t), reverse=True)) or (1.0,)
    except (TypeError, ValueError):
        lod_chain_t = (1.0,)

    print(f'\n[QuickTest] Reconstructing mesh (Poisson, depth={QUICK_MESH_DEPTH}) ...')
    t0 = time.time()
    paths, summary = export_all_formats(
        gauss, str(out_dir), base_name='quick',
        opacity_threshold=QUICK_OPACITY_THRESH, max_points=100_000,
        include_mesh=True, mesh_method='poisson',
        mesh_depth=QUICK_MESH_DEPTH, mesh_density_quantile=0.05,
        mesh_max_faces=200_000, fbx_up_axis=QUICK_FBX_UP,
        target_size_meters=QUICK_TARGET_SIZE, center_mode=QUICK_CENTER_MODE,
        lod_chain=lod_chain_t, srgb_colors=True,
    )
    print(f'[QuickTest] Mesh recon + export: {time.time() - t0:.1f}s')
    if 'bbox' in summary:
        bb = summary['bbox']
        print(f'[QuickTest] Mesh: {bb["n_vertices"]:,} verts / {bb["n_triangles"]:,} tris, '
              f'extent={bb["extent"]:.3f}m, center={bb["bbox_center"]}')
    if 'lod' in summary and len(summary['lod']) > 1:
        lod_summary = ', '.join(f'LOD{i}={s["faces"]:,}f' for i, s in enumerate(summary['lod']))
        print(f'[QuickTest] LOD chain: {lod_summary}')
    for fmt, p in sorted(paths.items()):
        if p.endswith(('.glb', '.obj', '.fbx', '.ply')):
            sz = pathlib.Path(p).stat().st_size / 1024
            display(FileLink(p))
            print(f'  {fmt.upper()}: {p} ({sz:.0f} KB)')

if torch.cuda.is_available():
    vram = torch.cuda.memory_allocated() / 1024**3
    print(f'\n[QuickTest] Final VRAM: {vram:.1f} GB')

print(f'\n[QuickTest] DONE. All outputs in {QUICK_OUTPUT_DIR}')


In [ ]:
# @title STEP 7 — Batch Generation from a .txt list
# @markdown Reads a text file of image paths (one per line) and generates a Gaussian set
# @markdown + mesh for each. Saves them to a per-batch folder. Use this for sweeping
# @markdown a folder of reference images.

import os, sys, time, pathlib, traceback
import torch
from PIL import Image

REPO_DIR = '/content/TripoSplat'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# @markdown **Batch parameters** (edit and run):
LIST_FILE = '/content/triposplat_runs/batch_list.txt'  # @param {type:"string"}
BATCH_STEPS = 15  # @param {type:"slider", min:5, max:50, step:1}
BATCH_NUM_GAUSSIANS = 65536  # @param {type:"slider", min:32768, max:262144, step:1024}
BATCH_SEED = 0  # @param {type:"integer"}
BATCH_DO_MESH = True  # @param {type:"boolean"}
BATCH_DO_RANDOM_SEED = False  # @param {type:"boolean"}
BATCH_OPACITY_THRESH = 0.05  # @param {type:"slider", min:0.01, max:0.5, step:0.01}
BATCH_TARGET_SIZE = 1.0  # @param {type:"slider", min:0.05, max:5.0, step:0.05, label:"Asset size (m)"}
BATCH_CENTER_MODE = 'base'  # @param ["base", "origin", "keep"]
BATCH_LOD_CHAIN = '1.0,0.5'  # @param {type:"string"}
BATCH_FBX_UP = 'Y'  # @param ["Y", "Z"]
OUT_ROOT = '/content/triposplat_runs/batch'  # @param {type:"string"}
MAX_ITEMS = 0  # @param {type:"integer"}
# @markdown *Set MAX_ITEMS > 0 to cap the batch (useful for testing).*

list_path = pathlib.Path(LIST_FILE)
if not list_path.exists():
    pngs = sorted(pathlib.Path('/content').rglob('*.png'))
    jpgs = sorted(pathlib.Path('/content').rglob('*.jpg')) + sorted(pathlib.Path('/content').rglob('*.jpeg'))
    examples = pngs + jpgs
    if not examples:
        raise SystemExit(f'No images found under /content. Put PNGs in a folder and set LIST_FILE.')
    list_path.parent.mkdir(parents=True, exist_ok=True)
    list_path.write_text('\n'.join(str(p) for p in examples) + '\n')
    print(f'[Batch] No list file found; bootstrapped {len(examples)} images into {list_path}')

lines = [l.strip() for l in list_path.read_text().splitlines()
         if l.strip() and not l.startswith('#')]
if MAX_ITEMS:
    lines = lines[:MAX_ITEMS]
print(f'[Batch] {len(lines)} image(s) queued.')

OUT_ROOT = pathlib.Path(OUT_ROOT)
OUT_ROOT.mkdir(parents=True, exist_ok=True)
batch_dir = OUT_ROOT / f'batch_{int(time.time())}'
batch_dir.mkdir(parents=True, exist_ok=True)

# Parse LOD chain
try:
    lod_chain_t = tuple(float(x.strip()) for x in BATCH_LOD_CHAIN.split(',') if x.strip())
    lod_chain_t = tuple(sorted(set(lod_chain_t), reverse=True)) or (1.0,)
except (TypeError, ValueError):
    lod_chain_t = (1.0,)

PIPE.load()
print(f'[Batch] Model loaded. LOD chain: {lod_chain_t}')

ok = 0
fail = 0
t_start = time.time()
for i, line in enumerate(lines, 1):
    try:
        p = pathlib.Path(line)
        if not p.exists():
            print(f'  [{i:03d}] SKIP (not found): {line}')
            fail += 1
            continue
        img = Image.open(p).convert('RGBA')
        seed = int.from_bytes(os.urandom(2), 'big') if BATCH_DO_RANDOM_SEED else BATCH_SEED
        t0 = time.time()
        gauss, prepared = PIPE.synthesize(
            img, seed=seed, steps=BATCH_STEPS,
            guidance_scale=3.0, shift=3.0,
            num_gaussians=BATCH_NUM_GAUSSIANS,
        )
        gen_dt = time.time() - t0
        n_g = gauss.get_xyz.shape[0]
        safe = ''.join(c if c.isalnum() else '_' for c in p.stem[:40]).strip('_') or f'item_{i:02d}'
        out_paths, summary = export_all_formats(
            gauss, str(batch_dir), base_name=f'{i:03d}_{safe}',
            opacity_threshold=BATCH_OPACITY_THRESH, max_points=100_000,
            include_mesh=BATCH_DO_MESH, mesh_method='poisson',
            mesh_depth=9, mesh_density_quantile=0.05,
            mesh_max_faces=200_000, fbx_up_axis=BATCH_FBX_UP,
            target_size_meters=BATCH_TARGET_SIZE if BATCH_DO_MESH else None,
            center_mode=BATCH_CENTER_MODE,
            lod_chain=lod_chain_t if BATCH_DO_MESH else (1.0,),
            srgb_colors=True,
        )
        n_formats = len(out_paths)
        ext = f" + LOD0-LOD{len(lod_chain_t)-1}" if BATCH_DO_MESH and len(lod_chain_t) > 1 else ''
        print(f'  [{i:03d}] {p.name} -> {n_g:,} Gaussians + {n_formats} files{ext} ({gen_dt:.1f}s)')
        ok += 1
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception as e:
        print(f'  [{i:03d}] EXCEPTION on {line}: {e}')
        traceback.print_exc()
        fail += 1

elapsed = time.time() - t_start
print(f'\n[Batch] DONE: {ok} OK / {fail} failed / {len(lines)} total in {elapsed:.1f}s -> {batch_dir}')
if ok > 0:
    print(f'[Batch] Tip: zip the folder with `!cd {batch_dir} && zip -r {batch_dir.name}.zip .` to download as a single file.')
